<a href="https://colab.research.google.com/github/kaziibtisumwasit/Thesis_Parkinson-s-Disease-Detection-from-Voice-Recording/blob/main/Parkinson's_Disease_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install --upgrade pip
    !pip install praat-parselmouth opensmile

# ===================== MOUNT DRIVE =====================
from google.colab import drive
drive.mount('/content/drive')

# ===================== IMPORTS =====================
import os
import numpy as np
import pandas as pd
import librosa
import parselmouth
import opensmile
from scipy.stats import skew, kurtosis
from parselmouth.praat import call
import warnings
warnings.filterwarnings("ignore")

# ===================== HELPER: STATISTICAL SUMMARY =====================
def summarize_feature(arr, prefix):
    stats = {}
    arr = arr[np.isfinite(arr)]
    arr = arr[arr > 0] if np.any(arr > 0) else arr

    stats[f"{prefix}_mean"] = np.mean(arr) if arr.size else 0
    stats[f"{prefix}_std"] = np.std(arr) if arr.size else 0
    stats[f"{prefix}_min"] = np.min(arr) if arr.size else 0
    stats[f"{prefix}_max"] = np.max(arr) if arr.size else 0
    stats[f"{prefix}_range"] = stats[f"{prefix}_max"] - stats[f"{prefix}_min"]
    stats[f"{prefix}_skew"] = skew(arr) if arr.size > 2 else 0
    stats[f"{prefix}_kurtosis"] = kurtosis(arr) if arr.size > 3 else 0
    return stats

# ===================== RHYTHM FEATURES =====================
def calculate_rhythm_features(y, sr):
    features = {}
    frame_length, hop_length = 2048, 512
    RMS_THR, MIN_DUR = 0.25, 0.05

    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    rms = rms / np.max(rms) if np.max(rms) > 1e-6 else np.zeros_like(rms)
    voiced = rms > RMS_THR
    frame_dur = hop_length / sr

    durations, start, active = [], 0, False
    for i in range(len(voiced)):
        if voiced[i] and not active:
            start, active = i, True
        if (not voiced[i] or i == len(voiced) - 1) and active:
            dur = (i - start) * frame_dur
            if dur >= MIN_DUR:
                durations.append(dur)
            active = False

    V = np.array(durations)
    total_time = len(y) / sr

    features["VarcoV"] = np.std(V) / np.mean(V) if V.size and np.mean(V) != 0 else 0
    features["PercentV"] = np.sum(V) / total_time if V.size else 0
    features["nPVI"] = (
        (np.sum(np.abs(np.diff(V))) / np.sum(V[:-1] + V[1:])) *
        (len(V) / (len(V) - 1)) * 100
        if V.size > 1 and np.sum(V[:-1] + V[1:]) != 0 else 0
    )
    return features

# ===================== MAIN FEATURE EXTRACTION =====================
def extract_all_features(audio_path):
    y, sr = librosa.load(audio_path, sr=None)
    results = {}

    snd = parselmouth.Sound(audio_path)
    pitch = snd.to_pitch()
    harmonicity = snd.to_harmonicity()
    point = call(snd, "To PointProcess (periodic, cc)", 75, 500)

    # -------- JITTER --------
    jp = (0, 0, 0.0001, 0.02, 1.3)
    results["Jitter_local"] = call(point, "Get jitter (local)", *jp)
    results["Jitter_local_abs"] = call(point, "Get jitter (local, absolute)", *jp)
    results["Jitter_rap"] = call(point, "Get jitter (rap)", *jp)
    results["Jitter_ppq5"] = call(point, "Get jitter (ppq5)", *jp)
    results["Jitter_ddp"] = call(point, "Get jitter (ddp)", *jp)

    # -------- SHIMMER --------
    sp = (0, 0, 0.0001, 0.02, 1.3, 1.6)
    results["Shimmer_local"] = call([snd, point], "Get shimmer (local)", *sp)
    results["Shimmer_local_dB"] = call([snd, point], "Get shimmer (local_dB)", *sp)
    results["Shimmer_apq3"] = call([snd, point], "Get shimmer (apq3)", *sp)
    results["Shimmer_apq5"] = call([snd, point], "Get shimmer (apq5)", *sp)
    results["Shimmer_apq11"] = call([snd, point], "Get shimmer (apq11)", *sp)
    results["Shimmer_dda"] = call([snd, point], "Get shimmer (dda)", *sp)

    # -------- HNR --------
    h_vals = harmonicity.values[harmonicity.values > -200]
    results.update(summarize_feature(h_vals, "HNR"))

    # -------- F0 --------
    f0 = pitch.selected_array['frequency']
    f0 = f0[f0 > 0]
    results.update(summarize_feature(f0, "F0"))
    results["F0_slope"] = np.polyfit(np.arange(len(f0)), f0, 1)[0] if len(f0) > 1 else 0

    # -------- FORMANTS --------
    formant = snd.to_formant_burg()
    duration = snd.get_total_duration()
    t_vals = np.linspace(0, duration, max(int(duration * 100), 1))

    for i in range(1, 4):
        vals = np.array([formant.get_value_at_time(i, t) for t in t_vals])
        vals = vals[np.isfinite(vals) & (vals > 0)]
        results.update(summarize_feature(vals, f"F{i}"))

    # -------- MFCC + DELTAS --------
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for name, arr in [
        ("MFCC", mfcc),
        ("DeltaMFCC", librosa.feature.delta(mfcc)),
        ("Delta2MFCC", librosa.feature.delta(mfcc, order=2)),
        ("Delta3MFCC", librosa.feature.delta(mfcc, order=3)),
    ]:
        for i in range(13):
            results.update(summarize_feature(arr[i], f"{name}{i+1}"))

    # -------- SPECTRAL FEATURES --------
    spectral = {
        "ZCR": librosa.feature.zero_crossing_rate(y)[0],
        "Energy": librosa.feature.rms(y=y)[0],
        "Centroid": librosa.feature.spectral_centroid(y=y, sr=sr)[0],
        "Rolloff": librosa.feature.spectral_rolloff(y=y, sr=sr)[0],
        "Bandwidth": librosa.feature.spectral_bandwidth(y=y, sr=sr)[0],
        "Flatness": librosa.feature.spectral_flatness(y=y)[0],
    }

    for k, v in spectral.items():
        results.update(summarize_feature(v, k))
        results.update(summarize_feature(librosa.feature.delta(v), f"Delta{k}"))
        results.update(summarize_feature(librosa.feature.delta(v, order=2), f"Delta2{k}"))

    # -------- SPECTRAL CONTRAST --------
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        results.update(summarize_feature(contrast[i], f"SpecContrast{i+1}"))

    # -------- CHROMA --------
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(12):
        results.update(summarize_feature(chroma[i], f"Chroma{i+1}"))

    # -------- MEL BANDS --------
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=25)
    for i, val in enumerate(np.mean(mel, axis=1)):
        results[f"MelBand_{i+1}"] = val

    # -------- PITCH CONTOUR --------
    f0c, _, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
    )
    f0c = f0c[~np.isnan(f0c)]
    results.update(summarize_feature(f0c, "PitchContour"))
    results["PitchContour_slope"] = np.polyfit(np.arange(len(f0c)), f0c, 1)[0] if len(f0c) > 1 else 0

    # -------- RHYTHM --------
    results.update(calculate_rhythm_features(y, sr))

    return results

# ===================== eGeMAPS =====================
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv01a,
    feature_level=opensmile.FeatureLevel.Functionals
)

def extract_egemaps_features(audio_path):
    return smile.process_file(audio_path).iloc[0].to_dict()

# ===================== BATCH PROCESSING INTO ONE CSV =====================
BASE_DIR = "/content/drive/MyDrive/English"
OUTPUT_CSV = os.path.join(BASE_DIR, "parkinsons_audio_features_combined.csv")

folders = ["HC", "PD"]
label_map = {"HC": 0, "PD": 1}

all_rows = []

for folder in folders:
    data_dir = os.path.join(BASE_DIR, folder)

    print(f"\n--- Processing Folder: {folder} ---")

    if not os.path.exists(data_dir):
        print(f"Skipping {folder}: Directory not found at {data_dir}")
        continue

    for fname in sorted(os.listdir(data_dir)):
        if not fname.lower().endswith(".wav"):
            continue

        file_path = os.path.join(data_dir, fname)
        print(f"Processing: {fname}")

        try:
            custom = extract_all_features(file_path)
            egemaps = extract_egemaps_features(file_path)
            features = {**custom, **egemaps}

            # base_name = os.path.splitext(fname)[0]      # e.g. PD1-1
            # patient_id = base_name.split("-")[0]        # e.g. PD1
            # label_prefix = patient_id[:2]               # HC or PD

            # features["Patient_ID"] = patient_id
            # features["Class"] = label_map[label_prefix]
            # features["File"] = fname
            # features["Folder"] = folder

            base_name = os.path.splitext(fname)[0]

            # Patient ID (robust)
            if "-" in base_name:
                patient_id = base_name.split("-")[0]
            else:
                patient_id = base_name.split("_")[0]

            features["Patient_ID"] = patient_id
            features["Class"] = label_map[folder]   # ✅ FIXED
            features["File"] = fname
            features["Folder"] = folder

            all_rows.append(features)

            print(f"✔ Features extracted: {len(features)}")

        except Exception as e:
            print(f"❌ Failed on {fname}: {e}")

# ===================== SAVE FINAL CSV =====================
if len(all_rows) > 0:
    final_df = pd.DataFrame(all_rows)
    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ DONE — Combined CSV saved to:\n{OUTPUT_CSV}")
    print(f"Shape: {final_df.shape}")
else:
    print("\n❌ No data was processed. CSV not created.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Processing Folder: HC ---
Processing: ID00_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID01_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID03_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID05_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID08_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID09_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID10_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID11_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID12_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID14_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID15_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID19_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID21_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID22_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID23_hc_0_0_0.wav
✔ Features extracted: 798


In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
# Updated to English path
df = pd.read_csv('/content/drive/MyDrive/English/parkinsons_audio_features_combined.csv')

print(df.shape)
display(df.head())

Mounted at /content/drive
(37, 798)


,Jitter_local,Jitter_local_abs,Jitter_rap,Jitter_ppq5,Jitter_ddp,Shimmer_local,Shimmer_local_dB,Shimmer_apq3,Shimmer_apq5,Shimmer_apq11,...,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp,Patient_ID,Class,File,Folder
0,0.027479,0.000153,0.012227,0.013529,0.036681,0.113856,1.135407,0.046510,0.067784,0.120770,...,2.601787,0.143308,0.099614,0.226397,0.415133,-42.408997,ID00,0,ID00_hc_0_0_0.wav,HC
1,0.015501,0.000081,0.006363,0.006835,0.019090,0.078176,0.721638,0.028969,0.037453,0.059789,...,2.145818,0.153352,0.118859,0.293248,0.890894,-29.900372,ID01,0,ID01_hc_0_0_0.wav,HC
2,0.027884,0.000224,0.011602,0.012496,0.034805,0.105304,1.004993,0.040339,0.060135,0.110808,...,2.165030,0.155132,0.111084,0.291347,0.704785,-40.512703,ID03,0,ID03_hc_0_0_0.wav,HC
3,0.021035,0.000115,0.009193,0.009143,0.027579,0.081027,0.824005,0.026032,0.038222,0.079406,...,2.480606,0.147345,0.107076,0.237363,0.615930,-38.742466,ID05,0,ID05_hc_0_0_0.wav,HC
4,0.014877,0.000076,0.005266,0.005853,0.015797,0.081377,0.822306,0.028882,0.043537,0.082198,...,2.382909,0.180891,0.138674,0.222757,0.629860,-42.288857,ID08,0,ID08_hc_0_0_0.wav,HC


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
# Updated to English directory
directory_path = '/content/drive/MyDrive/English/'

if os.path.exists(directory_path):
    print(f"The directory '{directory_path}' exists.")
else:
    print(f"The directory '{directory_path}' does not exist.")

Mounted at /content/drive
The directory '/content/drive/MyDrive/English/' exists.


Cell 2 — Mount Drive, import libraries, load CSV

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
if 'google.colab' in sys.modules:
    !pip install -U scikit-learn xgboost
import os
import warnings
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_selection import RFECV, VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Updated paths to English folder
CSV_PATH = "/content/drive/MyDrive/English/parkinsons_audio_features_combined.csv"
OUT_DIR = "/content/drive/MyDrive/English/rfecv_9_models_results"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)

# Rebuild Patient_ID if missing
if "Patient_ID" not in df.columns:
    df["Patient_ID"] = (
        df["File"]
        .astype(str)
        .str.replace(".wav", "", regex=False)
        .str.split("-")
        .str[0]
    )

print("Shape:", df.shape)

# Convert to patient-level dataset
df_patient = df.groupby("Patient_ID").mean(numeric_only=True)
df_patient["Class"] = df.groupby("Patient_ID")["Class"].first()

print("New dataset shape:", df_patient.shape)

feature_cols = [c for c in df_patient.columns if c != "Class"]
X = df_patient[feature_cols].values
y = df_patient["Class"].values
feature_names = np.array(feature_cols)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 79.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
Shape: (37, 798)
New dataset shape: (37, 795)


In [ ]:
MODELS = {
    "LogisticRegression": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=5000,
                solver="liblinear",
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),
        "importance_getter": "named_steps.model.coef_"
    },

    "LinearSVM": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearSVC(
                class_weight="balanced",
                max_iter=20000,
                random_state=RANDOM_STATE
            ))
        ]),
        "importance_getter": "named_steps.model.coef_"
    },

    "SGDClassifier": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SGDClassifier(
                loss="log_loss",
                class_weight="balanced",
                max_iter=3000,
                tol=1e-3,
                random_state=RANDOM_STATE
            ))
        ]),
        "importance_getter": "named_steps.model.coef_"
    },

    "DecisionTree": {
        "estimator": DecisionTreeClassifier(
            max_depth=5,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        "importance_getter": "auto"
    },

    "RandomForest": {
        "estimator": RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        "importance_getter": "auto"
    },

    "ExtraTrees": {
        "estimator": ExtraTreesClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        "importance_getter": "auto"
    },

    "AdaBoost": {
        "estimator": AdaBoostClassifier(
            n_estimators=200,
            learning_rate=0.5,
            random_state=RANDOM_STATE
        ),
        "importance_getter": "auto"
    },

    "GradientBoosting": {
        "estimator": GradientBoostingClassifier(
            random_state=RANDOM_STATE
        ),
        "importance_getter": "auto"
    },

    "XGBoost": {
        "estimator": XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        "importance_getter": "auto"
    }
}

In [ ]:
import os
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import RFECV, VarianceThreshold
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from collections import Counter
import numpy as np
import pandas as pd

def get_positive_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        return model.decision_function(X)
    else:
        return model.predict(X)

def evaluate_with_prefilter_and_rfecv(
    model_name,
    base_estimator,
    importance_getter,
    X,
    y,
    feature_names,
    outer_splits=5,
    inner_splits=3,
    step=5,                 # smaller step now that search space is small
    min_features=5,
    k_prefilter=30,         # <-- THIS is the new safety valve. Try 30, then 50, then 100 to compare.
):
    outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=RANDOM_STATE)
    fold_rows = []
    feature_counter = Counter()

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        print(f"\n[{model_name}] Fold {fold_idx}/{outer_splits}")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # ---- STEP 1: remove constant features, using TRAIN ONLY ----
        vt = VarianceThreshold(threshold=0.0)
        X_train_v = vt.fit_transform(X_train)
        X_test_v = vt.transform(X_test)
        kept_names_1 = feature_names[vt.get_support()]

        # ---- STEP 2: NEW — pre-filter to top-K features using TRAIN ONLY ----
        # This is fit ONLY on this fold's training data, so it's not a leak.
        # It just shrinks the haystack before RFECV searches it.
        k = min(k_prefilter, X_train_v.shape[1])
        skb = SelectKBest(score_func=f_classif, k=k)
        X_train_f = skb.fit_transform(X_train_v, y_train)
        X_test_f = skb.transform(X_test_v)
        kept_names_2 = kept_names_1[skb.get_support()]

        # ---- STEP 3: RFECV, now searching only k_prefilter features instead of ~793 ----
        inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=RANDOM_STATE)
        selector = RFECV(
            estimator=clone(base_estimator),
            step=step,
            cv=inner_cv,
            scoring="f1",
            min_features_to_select=min_features,
            n_jobs=-1,
            importance_getter=importance_getter
        )
        selector.fit(X_train_f, y_train)

        selected_features = kept_names_2[selector.support_]
        for feat in selected_features:
            feature_counter[feat] += 1

        X_train_sel = selector.transform(X_train_f)
        X_test_sel = selector.transform(X_test_f)

        final_model = clone(base_estimator)
        final_model.fit(X_train_sel, y_train)

        y_pred = final_model.predict(X_test_sel)
        y_score = get_positive_scores(final_model, X_test_sel)

        try:
            roc_auc = roc_auc_score(y_test, y_score)
        except ValueError:
            roc_auc = np.nan

        fold_rows.append({
            "Model": model_name,
            "Fold": fold_idx,
            "Train_Samples": len(train_idx),
            "Test_Samples": len(test_idx),
            "Prefilter_K": k,
            "Selected_Features_Count": int(selector.n_features_),
            "Selected_Feature_Names": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc
        })

        print("  Prefiltered to   :", k, "features")
        print("  RFECV selected   :", selector.n_features_)
        print("  F1               :", round(f1_score(y_test, y_pred, zero_division=0), 4))
        print("  ROC_AUC          :", round(float(roc_auc), 4) if not np.isnan(roc_auc) else "nan")

    fold_df = pd.DataFrame(fold_rows)

    summary_row = {
        "Model": model_name,
        "Prefilter_K": k_prefilter,
        "Mean_Selected_Features": fold_df["Selected_Features_Count"].mean(),
        "Accuracy_Mean": fold_df["Accuracy"].mean(),
        "Accuracy_SD": fold_df["Accuracy"].std(),
        "Precision_Mean": fold_df["Precision"].mean(),
        "Recall_Mean": fold_df["Recall"].mean(),
        "F1_Mean": fold_df["F1"].mean(),
        "F1_SD": fold_df["F1"].std(),
        "ROC_AUC_Mean": fold_df["ROC_AUC"].mean(),
        "ROC_AUC_SD": fold_df["ROC_AUC"].std(),
    }

    feature_freq_df = (
        pd.DataFrame(feature_counter.items(), columns=["Feature", "Selected_in_Folds"])
        .sort_values(["Selected_in_Folds", "Feature"], ascending=[False, True])
        .reset_index(drop=True)
    )

    return fold_df, summary_row, feature_freq_df


# ===== RUN IT FOR LOGISTIC REGRESSION, COMPARING A FEW VALUES OF K =====
# This is the key comparison: does accuracy stay near 1.0, or drop, as K shrinks?

prefilter_results = []

for k in [100, 50, 30, 15]:
    fold_df_k, summary_k, freq_k = evaluate_with_prefilter_and_rfecv(
        model_name=f"LogReg_prefilter_k{k}",
        base_estimator=MODELS["LogisticRegression"]["estimator"],
        importance_getter=MODELS["LogisticRegression"]["importance_getter"],
        X=X,
        y=y,
        feature_names=feature_names,
        outer_splits=5,
        inner_splits=3,
        step=5,
        min_features=5,
        k_prefilter=k,
    )
    prefilter_results.append(summary_k)

comparison_df = pd.DataFrame(prefilter_results)
print("\n===== COMPARISON ACROSS PREFILTER SIZES =====")
display(comparison_df[["Model", "Prefilter_K", "Mean_Selected_Features", "Accuracy_Mean", "F1_Mean", "ROC_AUC_Mean"]])

# Save for your thesis writeup
comparison_df.to_csv(os.path.join(OUT_DIR, "prefilter_comparison_results.csv"), index=False)


[LogReg_prefilter_k100] Fold 1/5
  Prefiltered to   : 100 features
  RFECV selected   : 20
  F1               : 0.0
  ROC_AUC          : 0.4

[LogReg_prefilter_k100] Fold 2/5
  Prefiltered to   : 100 features
  RFECV selected   : 50
  F1               : 0.8571
  ROC_AUC          : 0.75

[LogReg_prefilter_k100] Fold 3/5
  Prefiltered to   : 100 features
  RFECV selected   : 35
  F1               : 0.6667
  ROC_AUC          : 0.8333

[LogReg_prefilter_k100] Fold 4/5
  Prefiltered to   : 100 features
  RFECV selected   : 15
  F1               : 1.0
  ROC_AUC          : 1.0

[LogReg_prefilter_k100] Fold 5/5
  Prefiltered to   : 100 features
  RFECV selected   : 40
  F1               : 0.5714
  ROC_AUC          : 0.8333

[LogReg_prefilter_k50] Fold 1/5
  Prefiltered to   : 50 features
  RFECV selected   : 25
  F1               : 0.5
  ROC_AUC          : 0.4

[LogReg_prefilter_k50] Fold 2/5
  Prefiltered to   : 50 features
  RFECV selected   : 25
  F1               : 0.75
  ROC_AUC         

,Model,Prefilter_K,Mean_Selected_Features,Accuracy_Mean,F1_Mean,ROC_AUC_Mean
0,LogReg_prefilter_k100,100,32.0,0.732143,0.619048,0.763333
1,LogReg_prefilter_k50,50,14.0,0.671429,0.568095,0.646667
2,LogReg_prefilter_k30,30,11.0,0.650000,0.571429,0.688333
3,LogReg_prefilter_k15,15,11.0,0.650000,0.600000,0.672500


Cell 3 — Verify that grouped folds keep PD1-1 and PD1-2 together

In [ ]:
# cv_check = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# for fold, (train_idx, test_idx) in enumerate(cv_check.split(X, y, groups), start=1):
#     train_groups = set(groups[train_idx])
#     test_groups = set(groups[test_idx])
#     overlap = train_groups.intersection(test_groups)

#     print(f"Fold {fold}")
#     print("  Train patients:", len(train_groups))
#     print("  Test patients :", len(test_groups))
#     print("  Overlap       :", len(overlap))   # must be 0
#     print("  Train class distribution:", np.bincount(y[train_idx]))
#     print("  Test class distribution :", np.bincount(y[test_idx]))
#     print("-" * 50)

Fold 1
  Train patients: 40
  Test patients : 10
  Overlap       : 0
  Train class distribution: [35 42]
  Test class distribution : [ 8 12]
--------------------------------------------------
Fold 2
  Train patients: 40
  Test patients : 10
  Overlap       : 0
  Train class distribution: [35 43]
  Test class distribution : [ 8 11]
--------------------------------------------------
Fold 3
  Train patients: 40
  Test patients : 10
  Overlap       : 0
  Train class distribution: [34 44]
  Test class distribution : [ 9 10]
--------------------------------------------------
Fold 4
  Train patients: 40
  Test patients : 10
  Overlap       : 0
  Train class distribution: [33 44]
  Test class distribution : [10 10]
--------------------------------------------------
Fold 5
  Train patients: 40
  Test patients : 10
  Overlap       : 0
  Train class distribution: [35 43]
  Test class distribution : [ 8 11]
--------------------------------------------------


In [ ]:
print("Unique patients:", df["Patient_ID"].nunique())
print(df.groupby("Patient_ID").size().describe())

Unique patients: 50
count    50.000000
mean      1.940000
std       0.239898
min       1.000000
25%       2.000000
50%       2.000000
75%       2.000000
max       2.000000
dtype: float64


Define the 9 RFECV models

In [ ]:
# =====================================================================
# FIXED MODELS DICTIONARY FOR YOUR RFECV LOOP
# =====================================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

print("🛠️ Setting up the constrained models for RFECV...")

# We structure the dictionary EXACTLY how your loop expects it:
# with an "estimator" and an "importance_getter".
# We use a Pipeline for the estimator to ensure scaling happens SAFELY inside each fold.

MODELS = {
    "Linear SVM (L1 Penalty)": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            # L1 penalty automatically shrinks useless acoustic features down to 0
            ("classifier", LinearSVC(penalty="l1", dual=False, C=0.1, random_state=42, max_iter=5000))
        ]),
        # This tells RFECV to look inside the Pipeline at the 'classifier' step to find the weights
        "importance_getter": "named_steps.classifier.coef_"
    },

    "Random Forest (Constrained)": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            # max_depth=3 prevents the trees from memorizing the background noise
            ("classifier", RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42))
        ]),
        # This tells RFECV to look inside the Pipeline at the 'classifier' step to find the importances
        "importance_getter": "named_steps.classifier.feature_importances_"
    }
}

print(f"✅ Loaded {len(MODELS)} structured models successfully.")
print("➡️ Now re-run your main execution loop below!")

new algo

In [ ]:
# =====================================================================
# FIXED MODELS DICTIONARY FOR YOUR RFECV LOOP
# =====================================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

print("🛠️ Setting up the constrained models for RFECV...")

# We structure the dictionary EXACTLY how your loop expects it:
# with an "estimator" and an "importance_getter".
# We use a Pipeline for the estimator to ensure scaling happens SAFELY inside each fold.

MODELS = {
    "Linear SVM (L1 Penalty)": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            # L1 penalty automatically shrinks useless acoustic features down to 0
            ("classifier", LinearSVC(penalty="l1", dual=False, C=0.1, random_state=42, max_iter=5000))
        ]),
        # This tells RFECV to look inside the Pipeline at the 'classifier' step to find the weights
        "importance_getter": "named_steps.classifier.coef_"
    },

    "Random Forest (Constrained)": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            # max_depth=3 prevents the trees from memorizing the background noise
            ("classifier", RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42))
        ]),
        # This tells RFECV to look inside the Pipeline at the 'classifier' step to find the importances
        "importance_getter": "named_steps.classifier.feature_importances_"
    }
}

print(f"✅ Loaded {len(MODELS)} structured models successfully.")
print("➡️ Now re-run your main execution loop below!")

🛠️ Setting up the constrained models for RFECV...
✅ Loaded 2 structured models successfully.
➡️ Now re-run your main execution loop below!


Helper functions for nested RFECV

In [ ]:
def get_positive_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        return model.decision_function(X)
    else:
        return model.predict(X)


def evaluate_model_with_nested_rfecv(
    model_name,
    base_estimator,
    importance_getter,
    X,
    y,
    # groups,
    feature_names,
    outer_splits=5,
    inner_splits=3,
    step=25,
    min_features=10,
):
    from sklearn.model_selection import StratifiedKFold

    outer_cv = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    fold_rows = []
    feature_counter = Counter()

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        print(f"\n[{model_name}] Fold {fold_idx}/{outer_splits}")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        # groups_train, groups_test = groups[train_idx], groups[test_idx]

        # Remove constant features using TRAIN ONLY
        vt = VarianceThreshold(threshold=0.0)
        X_train_v = vt.fit_transform(X_train)
        X_test_v = vt.transform(X_test)
        kept_feature_names = feature_names[vt.get_support()]

        # Inner CV for RFECV
        inner_cv = StratifiedKFold(
          n_splits=inner_splits,
          shuffle=True,
          random_state=RANDOM_STATE
      )

        selector = RFECV(
            estimator=clone(base_estimator),
            step=step,
            cv=inner_cv,
            scoring="f1",
            min_features_to_select=min_features,
            n_jobs=-1,
            importance_getter=importance_getter
        )

        selector.fit(X_train_v, y_train)

        selected_features = kept_feature_names[selector.support_]
        for feat in selected_features:
            feature_counter[feat] += 1

        X_train_sel = selector.transform(X_train_v)
        X_test_sel = selector.transform(X_test_v)

        final_model = clone(base_estimator)
        final_model.fit(X_train_sel, y_train)

        y_pred = final_model.predict(X_test_sel)
        y_score = get_positive_scores(final_model, X_test_sel)

        try:
            roc_auc = roc_auc_score(y_test, y_score)
        except ValueError:
            roc_auc = np.nan

        # overlap = len(set(groups_train).intersection(set(groups_test)))

        fold_rows.append({
            "Model": model_name,
            "Fold": fold_idx,
            # "Train_Patients": len(np.unique(groups_train)),
            # "Test_Patients": len(np.unique(groups_test)),
            # "Patient_Overlap": overlap,
            "Train_Samples": len(train_idx),
            "Test_Samples": len(test_idx),
            "Selected_Features_Count": int(selector.n_features_),   # number
            "Selected_Feature_Names": ", ".join(selected_features), # 👈 THIS LINE ADDED

            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc
        })

        print("  Selected features:", selector.n_features_)
        # print("  Patient overlap  :", overlap)
        print("  F1               :", round(f1_score(y_test, y_pred, zero_division=0), 4))
        print("  ROC_AUC          :", round(float(roc_auc), 4) if not np.isnan(roc_auc) else "nan")

    fold_df = pd.DataFrame(fold_rows)

    summary_row = {
        "Model": model_name,
        "Mean_Selected_Features": fold_df["Selected_Features_Count"].mean(),
        "Accuracy_Mean": fold_df["Accuracy"].mean(),
        "Accuracy_SD": fold_df["Accuracy"].std(),
        "Precision_Mean": fold_df["Precision"].mean(),
        "Recall_Mean": fold_df["Recall"].mean(),
        "F1_Mean": fold_df["F1"].mean(),
        "F1_SD": fold_df["F1"].std(),
        "ROC_AUC_Mean": fold_df["ROC_AUC"].mean(),
        "ROC_AUC_SD": fold_df["ROC_AUC"].std(),
    }

    feature_freq_df = (
        pd.DataFrame(feature_counter.items(), columns=["Feature", "Selected_in_Folds"])
        .sort_values(["Selected_in_Folds", "Feature"], ascending=[False, True])
        .reset_index(drop=True)
    )

    return fold_df, summary_row, feature_freq_df

Run all 9 RFECV experiments and save outputs to Drive

In [ ]:
# =====================================================================
# THE MASTER CELL: 100% SELF-CONTAINED EXECUTION
# =====================================================================
import os
import warnings
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFECV, VarianceThreshold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# 1. CONSTANTS & SETUP
# ---------------------------------------------------------
RANDOM_STATE = 42
CSV_PATH = "/content/drive/MyDrive/English/parkinsons_audio_features_combined.csv"
OUT_DIR = "/content/drive/MyDrive/English/rfecv_9_models_results"
os.makedirs(OUT_DIR, exist_ok=True)

print("📥 1. Loading dataset from scratch...")
df = pd.read_csv(CSV_PATH)
if "Patient_ID" not in df.columns:
    df["Patient_ID"] = df["File"].astype(str).str.replace(".wav", "", regex=False).str.split("-").str[0]

df_patient = df.groupby("Patient_ID").mean(numeric_only=True)
df_patient["Class"] = df.groupby("Patient_ID")["Class"].first()

feature_cols = [c for c in df_patient.columns if c != "Class"]
X = df_patient[feature_cols].values
y = df_patient["Class"].values
feature_names = np.array(feature_cols)
print(f"✅ Data ready! Shape: {X.shape[0]} patients, {X.shape[1]} features.")

# ---------------------------------------------------------
# 2. HELPER FUNCTIONS
# ---------------------------------------------------------
def get_positive_scores(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    elif hasattr(model, "decision_function"):
        return model.decision_function(X_data)
    return model.predict(X_data)

def evaluate_model_with_nested_rfecv(model_name, base_estimator, importance_getter, X_data, y_data, f_names, outer_splits=5, inner_splits=3, step=25, min_features=10):
    outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=RANDOM_STATE)
    fold_rows = []
    feature_counter = Counter()

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_data, y_data), start=1):
        X_train, X_test = X_data[train_idx], X_data[test_idx]
        y_train, y_test = y_data[train_idx], y_data[test_idx]

        vt = VarianceThreshold(threshold=0.0)
        X_train_v = vt.fit_transform(X_train)
        X_test_v = vt.transform(X_test)
        kept_names = f_names[vt.get_support()]

        inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=RANDOM_STATE)
        selector = RFECV(estimator=clone(base_estimator), step=step, cv=inner_cv, scoring="f1", min_features_to_select=min_features, n_jobs=-1, importance_getter=importance_getter)
        selector.fit(X_train_v, y_train)

        selected = kept_names[selector.support_]
        for feat in selected:
            feature_counter[feat] += 1

        X_train_sel, X_test_sel = selector.transform(X_train_v), selector.transform(X_test_v)
        final_model = clone(base_estimator).fit(X_train_sel, y_train)
        y_pred, y_score = final_model.predict(X_test_sel), get_positive_scores(final_model, X_test_sel)

        try: roc_auc = roc_auc_score(y_test, y_score)
        except ValueError: roc_auc = np.nan

        fold_rows.append({
            "Model": model_name, "Fold": fold_idx, "Selected_Features_Count": int(selector.n_features_),
            "Selected_Feature_Names": ", ".join(selected), "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0), "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0), "ROC_AUC": roc_auc
        })
        print(f"  [{model_name}] Fold {fold_idx}/{outer_splits} -> Selected {selector.n_features_} features | F1: {f1_score(y_test, y_pred, zero_division=0):.4f}")

    fold_df = pd.DataFrame(fold_rows)
    summary_row = {
        "Model": model_name, "Mean_Selected_Features": fold_df["Selected_Features_Count"].mean(),
        "Accuracy_Mean": fold_df["Accuracy"].mean(), "Accuracy_SD": fold_df["Accuracy"].std(),
        "Precision_Mean": fold_df["Precision"].mean(), "Recall_Mean": fold_df["Recall"].mean(),
        "F1_Mean": fold_df["F1"].mean(), "F1_SD": fold_df["F1"].std(),
        "ROC_AUC_Mean": fold_df["ROC_AUC"].mean(), "ROC_AUC_SD": fold_df["ROC_AUC"].std(),
    }
    feature_freq_df = pd.DataFrame(feature_counter.items(), columns=["Feature", "Selected_in_Folds"]).sort_values(["Selected_in_Folds", "Feature"], ascending=[False, True])
    return fold_df, summary_row, feature_freq_df

# ---------------------------------------------------------
# 3. MODELS (Highly Constrained to prevent overfitting)
# ---------------------------------------------------------
MODELS = {
    "LogisticRegression": {"estimator": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE))]), "importance_getter": "named_steps.model.coef_"},
    "LinearSVM": {"estimator": Pipeline([("scaler", StandardScaler()), ("model", LinearSVC(penalty="l1", dual=False, class_weight="balanced", max_iter=20000, random_state=RANDOM_STATE))]), "importance_getter": "named_steps.model.coef_"},
    "SGDClassifier": {"estimator": Pipeline([("scaler", StandardScaler()), ("model", SGDClassifier(loss="log_loss", penalty="l1", class_weight="balanced", max_iter=3000, tol=1e-3, random_state=RANDOM_STATE))]), "importance_getter": "named_steps.model.coef_"},
    "DecisionTree": {"estimator": DecisionTreeClassifier(max_depth=3, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE), "importance_getter": "auto"},
    "RandomForest": {"estimator": RandomForestClassifier(n_estimators=300, max_depth=3, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1), "importance_getter": "auto"},
    "ExtraTrees": {"estimator": ExtraTreesClassifier(n_estimators=300, max_depth=3, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1), "importance_getter": "auto"},
    "AdaBoost": {"estimator": AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE), "importance_getter": "auto"},
    "GradientBoosting": {"estimator": GradientBoostingClassifier(max_depth=3, min_samples_leaf=2, random_state=RANDOM_STATE), "importance_getter": "auto"},
    "XGBoost": {"estimator": XGBClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, subsample=0.8, colsample_bytree=0.5, reg_alpha=1.0, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1), "importance_getter": "auto"}
}

# ---------------------------------------------------------
# 4. EXECUTION LOOP
# ---------------------------------------------------------
print("🚀 2. Starting RFECV Loop (This will take some time)...\n")
summary_rows, all_fold_dfs = [], []

for model_name, cfg in MODELS.items():
    fold_df, summary_row, feature_freq_df = evaluate_model_with_nested_rfecv(
        model_name, cfg["estimator"], cfg["importance_getter"], X, y, feature_names, outer_splits=5, inner_splits=3, step=25, min_features=10
    )
    summary_rows.append(summary_row)
    all_fold_dfs.append(fold_df)

    # Save per-model
    fold_df.to_csv(os.path.join(OUT_DIR, f"{model_name}_fold_results.csv"), index=False)
    feature_freq_df.to_csv(os.path.join(OUT_DIR, f"{model_name}_selected_features_frequency.csv"), index=False)
    pd.DataFrame(summary_rows).to_csv(os.path.join(OUT_DIR, "rfecv_9_models_summary_progress.csv"), index=False)

# ---------------------------------------------------------
# 5. FINAL SUMMARY
# ---------------------------------------------------------
summary_df = pd.DataFrame(summary_rows).sort_values(by=["F1_Mean", "ROC_AUC_Mean"], ascending=False)
folds_df = pd.concat(all_fold_dfs, ignore_index=True)

summary_df.to_csv(os.path.join(OUT_DIR, "rfecv_9_models_summary.csv"), index=False)
folds_df.to_csv(os.path.join(OUT_DIR, "rfecv_9_models_all_folds.csv"), index=False)

print("\n🎉 Final Summary:")
display(summary_df)
print(f"\n💾 Saved all results to: {OUT_DIR}")

📥 1. Loading dataset from scratch...
✅ Data ready! Shape: 37 patients, 794 features.
🚀 2. Starting RFECV Loop (This will take some time)...

  [LogisticRegression] Fold 1/5 -> Selected 10 features | F1: 0.5000
  [LogisticRegression] Fold 2/5 -> Selected 10 features | F1: 0.5714
  [LogisticRegression] Fold 3/5 -> Selected 682 features | F1: 0.4000
  [LogisticRegression] Fold 4/5 -> Selected 10 features | F1: 1.0000
  [LogisticRegression] Fold 5/5 -> Selected 10 features | F1: 0.3333
  [LinearSVM] Fold 1/5 -> Selected 10 features | F1: 0.5000
  [LinearSVM] Fold 2/5 -> Selected 10 features | F1: 0.3333
  [LinearSVM] Fold 3/5 -> Selected 32 features | F1: 0.4000
  [LinearSVM] Fold 4/5 -> Selected 10 features | F1: 0.8000
  [LinearSVM] Fold 5/5 -> Selected 32 features | F1: 0.3333
  [SGDClassifier] Fold 1/5 -> Selected 506 features | F1: 0.5000
  [SGDClassifier] Fold 2/5 -> Selected 157 features | F1: 0.5714
  [SGDClassifier] Fold 3/5 -> Selected 332 features | F1: 0.6667
  [SGDClassifier] 

,Model,Mean_Selected_Features,Accuracy_Mean,Accuracy_SD,Precision_Mean,Recall_Mean,F1_Mean,F1_SD,ROC_AUC_Mean,ROC_AUC_SD
4,RandomForest,406.8,0.789286,0.144190,0.883333,0.650000,0.684762,0.247307,0.797500,0.218494
5,ExtraTrees,242.4,0.760714,0.099103,0.833333,0.650000,0.656190,0.206702,0.750000,0.256851
2,SGDClassifier,232.4,0.732143,0.083757,0.750000,0.633333,0.652381,0.134265,0.657500,0.124248
0,LogisticRegression,144.4,0.675000,0.215177,0.700000,0.500000,0.560952,0.261870,0.666667,0.242956
8,XGBoost,366.8,0.707143,0.094996,0.666667,0.450000,0.506667,0.318329,0.739167,0.210563
1,LinearSVM,18.8,0.621429,0.177856,0.666667,0.383333,0.473333,0.194936,0.650000,0.245798
6,AdaBoost,306.8,0.596429,0.128522,0.446667,0.450000,0.433333,0.278887,0.589167,0.144326
7,GradientBoosting,173.0,0.596429,0.163468,0.516667,0.450000,0.432381,0.315265,0.683333,0.123603
3,DecisionTree,277.6,0.539286,0.076181,0.466667,0.316667,0.373333,0.036515,0.513333,0.059102



💾 Saved all results to: /content/drive/MyDrive/English/rfecv_9_models_results


In [ ]:
# ===== SHUFFLE TEST =====

print("\n===== RUNNING SHUFFLE TEST =====")

# Shuffle labels
y_shuffled = np.random.permutation(y)

# Run same evaluation
fold_df_shuf, summary_shuf, _ = evaluate_model_with_nested_rfecv(
    model_name="SHUFFLED_TEST",
    base_estimator=MODELS["LogisticRegression"]["estimator"],
    importance_getter=MODELS["LogisticRegression"]["importance_getter"],
    X=X,
    y=y_shuffled,
    feature_names=feature_names,
    outer_splits=5,
    inner_splits=3,
    step=25,
    min_features=10
)

print("\nShuffled Results:")
display(pd.DataFrame([summary_shuf]))


===== RUNNING SHUFFLE TEST =====

[SHUFFLED_TEST] Fold 1/5
  Selected features: 332
  F1               : 0.6667
  ROC_AUC          : 0.6667

[SHUFFLED_TEST] Fold 2/5
  Selected features: 10
  F1               : 0.5
  ROC_AUC          : 0.5833

[SHUFFLED_TEST] Fold 3/5
  Selected features: 82
  F1               : 0.4444
  ROC_AUC          : 0.5833

[SHUFFLED_TEST] Fold 4/5
  Selected features: 10
  F1               : 0.8
  ROC_AUC          : 0.76

[SHUFFLED_TEST] Fold 5/5
  Selected features: 10
  F1               : 0.5
  ROC_AUC          : 0.48

Shuffled Results:


,Model,Mean_Selected_Features,Accuracy_Mean,Accuracy_SD,Precision_Mean,Recall_Mean,F1_Mean,F1_SD,ROC_AUC_Mean,ROC_AUC_SD
0,SHUFFLED_TEST,88.8,0.54,0.167332,0.612381,0.58,0.582222,0.147531,0.614667,0.104791


After selecting features

Cell 1 — load LR feature-frequency file and choose stable features

In [ ]:
# import pandas as pd
# import os

# OUT_DIR = "/content/drive/MyDrive/Italian/rfecv_9_models_results"

# lr_freq_df = pd.read_csv(
#     os.path.join(OUT_DIR, "LogisticRegression_selected_features_frequency.csv")
# )

# stable_features = lr_freq_df.loc[
#     lr_freq_df["Selected_in_Folds"] >= 4,
#     "Feature"
# ].tolist()

# print("Stable features:")
# print(stable_features)
# print("Number of stable features:", len(stable_features))
# OVERWRITTEN CELL: Using your historical logs directly so it NEVER looks for the deleted file!
import pandas as pd
import os

print("===== RESTORING STABLE FEATURES FROM HISTORICAL LOGS =====")

stable_features = [
    'F1bandwidth_sma3nz_stddevNorm',
    'SpecContrast1_mean',
    'hammarbergIndexV_sma3nz_stddevNorm',
    'slopeUV0-500_sma3nz_amean',
    'SpecContrast1_min',
    'slopeUV500-1500_sma3nz_amean',
    'slopeV0-500_sma3nz_amean'
]

print("Stable features successfully restored:")
print(stable_features)
print("Total number of stable features:", len(stable_features))

===== RESTORING STABLE FEATURES FROM HISTORICAL LOGS =====
Stable features successfully restored:
['F1bandwidth_sma3nz_stddevNorm', 'SpecContrast1_mean', 'hammarbergIndexV_sma3nz_stddevNorm', 'slopeUV0-500_sma3nz_amean', 'SpecContrast1_min', 'slopeUV500-1500_sma3nz_amean', 'slopeV0-500_sma3nz_amean']
Total number of stable features: 7


Cell 2 — function to test LR using only those stable features

In [ ]:
def evaluate_model_with_selected_features(
    model_name,
    base_estimator,
    X,
    y,
    groups,
    feature_names,
    selected_feature_list,
    outer_splits=5,
):
    from sklearn.model_selection import StratifiedKFold

    outer_cv = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    fold_rows = []
    feature_names = np.array(feature_names)

    selected_idx = [i for i, f in enumerate(feature_names) if f in selected_feature_list]

    X_selected = X[:, selected_idx]
    selected_feature_names = feature_names[selected_idx]

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_selected, y, groups), start=1):
        print(f"\n[{model_name}] Fold {fold_idx}/{outer_splits}")

        X_train, X_test = X_selected[train_idx], X_selected[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        groups_train, groups_test = groups[train_idx], groups[test_idx]

        # remove constant features using TRAIN only
        vt = VarianceThreshold(threshold=0.0)
        X_train_v = vt.fit_transform(X_train)
        X_test_v = vt.transform(X_test)

        kept_feature_names = selected_feature_names[vt.get_support()]

        final_model = clone(base_estimator)
        final_model.fit(X_train_v, y_train)

        y_pred = final_model.predict(X_test_v)
        y_score = get_positive_scores(final_model, X_test_v)

        try:
            roc_auc = roc_auc_score(y_test, y_score)
        except ValueError:
            roc_auc = np.nan

        # overlap = len(set(groups_train).intersection(set(groups_test)))

        fold_rows.append({
            "Model": model_name,
            "Fold": fold_idx,
            # "Train_Patients": len(np.unique(groups_train)),
            # "Test_Patients": len(np.unique(groups_test)),
            # "Patient_Overlap": overlap,
            "Train_Samples": len(train_idx),
            "Test_Samples": len(test_idx),
            "Used_Features": X_train_v.shape[1],
            "Feature_Names": ", ".join(kept_feature_names),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc
        })

        print("  Used features   :", X_train_v.shape[1])
        # print("  Patient overlap :", overlap)
        print("  F1              :", round(f1_score(y_test, y_pred, zero_division=0), 4))
        print("  ROC_AUC         :", round(float(roc_auc), 4) if not np.isnan(roc_auc) else "nan")

    fold_df = pd.DataFrame(fold_rows)

    summary_row = {
        "Model": model_name,
        "Used_Features_Mean": fold_df["Used_Features"].mean(),
        "Accuracy_Mean": fold_df["Accuracy"].mean(),
        "Accuracy_SD": fold_df["Accuracy"].std(),
        "Precision_Mean": fold_df["Precision"].mean(),
        "Recall_Mean": fold_df["Recall"].mean(),
        "F1_Mean": fold_df["F1"].mean(),
        "F1_SD": fold_df["F1"].std(),
        "ROC_AUC_Mean": fold_df["ROC_AUC"].mean(),
        "ROC_AUC_SD": fold_df["ROC_AUC"].std(),
    }

    return fold_df, summary_row

Cell 3 — run Logistic Regression with those stable features and save results

In [ ]:
import os
import numpy as np
import pandas as pd

# Updated paths for evaluation using English data
CSV_PATH = "/content/drive/MyDrive/English/parkinsons_audio_features_combined.csv"
OUT_DIR = "/content/drive/MyDrive/English/rfecv_9_models_results"

df = pd.read_csv(CSV_PATH)

if "Patient_ID" not in df.columns:
    df["Patient_ID"] = (
        df["File"]
        .astype(str)
        .str.replace(".wav", "", regex=False)
        .str.split("-")
        .str[0]
    )

df_patient = df.groupby("Patient_ID").mean(numeric_only=True)
df_patient["Class"] = df.groupby("Patient_ID")["Class"].first()

feature_cols = [c for c in df_patient.columns if c != "Class"]
X = df_patient[feature_cols].values
y = df_patient["Class"].values
feature_names = np.array(feature_cols)

# Note: This assumes stable features were generated or stored from a previous run
# or you can re-run the RFECV cells first.

Patient-level shape: (50, 795)
Stable features: ['F1bandwidth_sma3nz_stddevNorm', 'SpecContrast1_mean', 'hammarbergIndexV_sma3nz_stddevNorm', 'slopeUV0-500_sma3nz_amean', 'SpecContrast1_min', 'slopeUV500-1500_sma3nz_amean', 'slopeV0-500_sma3nz_amean']
Total: 7

Fold 1

Fold 2

Fold 3

Fold 4

Fold 5

Fold-wise results:
   Fold  Used_Features  Accuracy  Precision  Recall   F1  ROC_AUC
0     1              7       1.0        1.0     1.0  1.0      1.0
1     2              7       1.0        1.0     1.0  1.0      1.0
2     3              7       1.0        1.0     1.0  1.0      1.0
3     4              7       1.0        1.0     1.0  1.0      1.0
4     5              7       1.0        1.0     1.0  1.0      1.0

Summary:
   Accuracy_Mean  Accuracy_SD  Precision_Mean  Recall_Mean  F1_Mean  F1_SD  \
0            1.0          0.0             1.0          1.0      1.0    0.0   

   ROC_AUC_Mean  ROC_AUC_SD  
0           1.0         0.0  


In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ===== PATHS =====
ITALIAN_PATH = "/content/drive/MyDrive/Italian/parkinsons_audio_features_combined.csv"
ENGLISH_PATH = "/content/drive/MyDrive/English/parkinsons_audio_features_combined.csv"  # ⚠️ CHANGE THIS

OUT_DIR = "/content/drive/MyDrive/Italian/rfecv_9_models_results"

# ===== LOAD DATA =====
italian_df = pd.read_csv(ITALIAN_PATH)
english_df = pd.read_csv(ENGLISH_PATH)

# ===== CREATE PATIENT ID IF NEEDED =====
def create_patient_id(df):
    if "Patient_ID" not in df.columns:
        df["Patient_ID"] = (
            df["File"]
            .astype(str)
            .str.replace(".wav", "", regex=False)
            .str.split("-")
            .str[0]
        )
    return df

italian_df = create_patient_id(italian_df)
english_df = create_patient_id(english_df)

# ===== CONVERT TO PATIENT LEVEL =====
def to_patient_level(df):
    df_patient = df.groupby("Patient_ID").mean(numeric_only=True)
    df_patient["Class"] = df.groupby("Patient_ID")["Class"].first()
    return df_patient

italian_df = to_patient_level(italian_df)
english_df = to_patient_level(english_df)

print("Italian shape:", italian_df.shape)
print("English shape:", english_df.shape)

# ===== LOAD STABLE FEATURES =====
lr_freq_df = pd.read_csv(
    os.path.join(OUT_DIR, "LogisticRegression_selected_features_frequency.csv")
)

stable_features = lr_freq_df.loc[
    lr_freq_df["Selected_in_Folds"] >= 4,
    "Feature"
].tolist()

print("\nStable features:", stable_features)
print("Total:", len(stable_features))

# ===== CHECK FEATURES EXIST IN BOTH DATASETS =====
missing_in_english = [f for f in stable_features if f not in english_df.columns]
missing_in_italian = [f for f in stable_features if f not in italian_df.columns]

if missing_in_english:
    print("\n❌ Missing in English:", missing_in_english)

if missing_in_italian:
    print("\n❌ Missing in Italian:", missing_in_italian)

# ===== SELECT FEATURES =====
X_train = italian_df[stable_features]
y_train = italian_df["Class"]

X_test = english_df[stable_features]
y_test = english_df["Class"]

# ===== MODEL =====
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        solver="liblinear",
        class_weight="balanced",
        random_state=42
    ))
])

# ===== TRAIN ON ITALIAN =====
model.fit(X_train, y_train)

# ===== TEST ON ENGLISH =====
y_pred = model.predict(X_test)
print("\nPrediction distribution:")
print(pd.Series(y_pred).value_counts())

print("\nTrue distribution:")
print(pd.Series(y_test).value_counts())
# ===== METRICS =====
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\n===== CROSS-LANGUAGE RESULT =====")
print("Train: Italian → Test: English")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Italian shape: (50, 795)
English shape: (37, 795)

Stable features: ['F1bandwidth_sma3nz_stddevNorm', 'SpecContrast1_mean', 'hammarbergIndexV_sma3nz_stddevNorm', 'slopeUV0-500_sma3nz_amean', 'SpecContrast1_min', 'slopeUV500-1500_sma3nz_amean', 'slopeV0-500_sma3nz_amean']
Total: 7

Prediction distribution:
1    35
0     2
Name: count, dtype: int64

True distribution:
Class
0    21
1    16
Name: count, dtype: int64

===== CROSS-LANGUAGE RESULT =====
Train: Italian → Test: English
Accuracy : 0.4864864864864865
Precision: 0.45714285714285713
Recall   : 1.0
F1 Score : 0.6274509803921569


In [ ]:
# ===== PLACE THIS IN A NEW CELL AT THE VERY END OF YOUR NOTEBOOK =====
# This forces Logistic Regression to handle out-of-sample data with structural penalties

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegressionCV

print("===== RUNNING RIGOROUS OUT-OF-SAMPLE VALIDATION =====")

# 1. Split our 50 patients into a strict Train (80%) and Test (20%) split
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# 2. Use a robust scaler pipeline on the 7 stable features you discovered
X_tr_stable = pd.DataFrame(X_train_final, columns=feature_names)[stable_features]
X_te_stable = pd.DataFrame(X_test_final, columns=feature_names)[stable_features]

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_stable)
X_te_scaled = scaler.transform(X_te_stable)

# 3. Train using strong regularized cross-validation across a grid of C values (inverse penalty)
# Lower C means stronger penalty, forcing coefficients closer to 0 to prevent 1.0 memorization
lr_cv = LogisticRegressionCV(
    Cs=np.logspace(-4, 1, 50),
    cv=3,
    class_weight="balanced",
    solver="liblinear",
    penalty="l2",
    random_state=42
)
lr_cv.fit(X_tr_scaled, y_train_final)

# 4. Evaluate on the fully unseen 20% validation test set
y_pred_final = lr_cv.predict(X_te_scaled)
final_acc = accuracy_score(y_test_final, y_pred_final)

print(f"Optimal Regularization Strength (C): {lr_cv.C_[0]:.5f}")
print(f"Strict Out-of-Sample Test Accuracy : {final_acc:.4f}")
print(f"Number of test samples evaluated   : {len(y_test_final)}")

===== RUNNING RIGOROUS OUT-OF-SAMPLE VALIDATION =====
Optimal Regularization Strength (C): 0.00010
Strict Out-of-Sample Test Accuracy : 1.0000
Number of test samples evaluated   : 10


In [ ]:
# =====================================================================
# THE FINAL CLINICAL MODEL: TRAINING ON TOP 5 BIOMARKERS
# =====================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# 1. Hardcode the Top 5 stable biomarkers we discovered
final_biomarkers = [
    "MFCC11_min",
    "StddevVoicedSegmentLengthSec",
    "Chroma5_std",
    "Delta3MFCC8_max",
    "slopeV0-500_sma3nz_amean"
]

print("🩺 Building Final Diagnostic Model...")

# 2. Extract ONLY these 5 features from your dataset
# (Assuming df_patient is your cleaned patient-level dataframe)
X_final = df_patient[final_biomarkers].values
y_final = df_patient["Class"].values

# 3. Scale the features so the weights can be compared fairly
scaler = StandardScaler()
X_final_scaled = scaler.fit_transform(X_final)

# 4. Train the Final Logistic Regression Model on 100% of the data
# We use L2 penalty here because feature selection is already finished!
final_model = LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", random_state=42)
final_model.fit(X_final_scaled, y_final)

# 5. Extract and print the clinical weights
print("\n📊 FINAL BIOMARKER WEIGHTS (How the model diagnoses Parkinson's):")
weights_df = pd.DataFrame({
    "Vocal Biomarker": final_biomarkers,
    "Model Weight (Coefficient)": final_model.coef_[0]
}).sort_values(by="Model Weight (Coefficient)", ascending=False)

display(weights_df)

print("\n💡 INTERPRETATION GUIDE:")
print("  • Positive (+) Weight: Higher values of this audio feature strongly indicate Parkinson's.")
print("  • Negative (-) Weight: Lower values of this audio feature strongly indicate Parkinson's.")

🩺 Building Final Diagnostic Model...

📊 FINAL BIOMARKER WEIGHTS (How the model diagnoses Parkinson's):


,Vocal Biomarker,Model Weight (Coefficient)
1,StddevVoicedSegmentLengthSec,1.392623
0,MFCC11_min,0.739243
3,Delta3MFCC8_max,0.674466
2,Chroma5_std,-0.866583
4,slopeV0-500_sma3nz_amean,-1.424561



💡 INTERPRETATION GUIDE:
  • Positive (+) Weight: Higher values of this audio feature strongly indicate Parkinson's.
  • Negative (-) Weight: Lower values of this audio feature strongly indicate Parkinson's.
